# **TaniMol: 01 - Preprocessing Pipeline**

This notebook loads raw bioactivity data from the ChEMBL database for five DNA repair protein inhibitor targets, cleans and standardizes the molecules, and produces a processed dataset ready for fingerprinting and similarity analysis.

**Targets:** PARP1, PARP2, ATR, ATM, DNA-PKcs  
**Source:** ChEMBL v36 (local SQLite database)  
**Output:** `data/processed/cleaned_activities.csv`

In [2]:
from src.config import DB_PATH, TARGETS, MIN_CONFIDENCE, ACTIVITY_TYPES, ACTIVITY_UNITS, OUTPUT_PATH
from src.preprocessing import (
    fetch_activity_data,
    drop_missing_values,
    validate_smiles,
    standardize_molecules,
    deduplicate,
    compute_pic50,
    save_cleaned_data,
)

from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

### **1. Data Acquisition**

Query the local ChEMBL SQLite database for bioactivity records (IC50, Ki) across all configured targets. Only records with confidence score ≥ 7 and units in nM are included.

In [3]:
df = fetch_activity_data(DB_PATH, TARGETS, MIN_CONFIDENCE, ACTIVITY_TYPES, ACTIVITY_UNITS)
print(f"Downloaded {len(df)} rows")
df.head()

Downloaded 16869 rows


,molecule_chembl_id,canonical_smiles,target_chembl_id,standard_type,standard_value,standard_units,pchembl_value
0,CHEMBL104450,O=c1cc(-c2cccc(-c3ccc(O)cc3)c2)sc(N2CCOCC2)c1,CHEMBL3142,IC50,350.0,nM,6.46
1,CHEMBL98350,O=c1cc(N2CCOCC2)oc2c(-c3ccccc3)cccc12,CHEMBL3142,IC50,1400.0,nM,5.85
2,CHEMBL276746,O=c1cc(-c2ccccc2)sc(N2CCOCC2)c1,CHEMBL3142,IC50,720.0,nM,6.14
3,CHEMBL102529,CC(C)(C)c1ccc(-c2cc(=O)cc(N3CCOCC3)o2)cc1,CHEMBL3142,IC50,480.0,nM,6.32
4,CHEMBL105613,O=c1cc(-c2ccccc2)oc(N2CCOCC2)c1,CHEMBL3142,IC50,1100.0,nM,5.96


### **2. Drop Missing Values**

Remove rows where `standard_value` (IC50/Ki measurement) is missing — these rows cannot be used for activity analysis.

In [4]:
rows_before = len(df)
df = drop_missing_values(df)
print(f"Dropped {rows_before - len(df)} rows with missing standard_value.")

Dropped 5 rows with missing standard_value.


### **3. SMILES Validation**

Parse each SMILES string with RDKit and remove any that cannot be interpreted as a valid molecule. ChEMBL data is well-curated, so we expect very few (if any) invalid entries.

In [5]:
rows_before = len(df)
df = validate_smiles(df)
print(f"Dropped {rows_before - len(df)} rows with invalid SMILES.")

Dropped 0 rows with invalid SMILES.


### **4. Molecule Standardization**

Standardize molecular representations to ensure consistent comparisons:
- **Salt stripping** — remove counter-ions, keep only the largest (active) fragment
- **Charge neutralization** — convert ionized forms to neutral
- **Tautomer canonicalization** — choose a single canonical tautomeric form

This step does not remove rows — it modifies the SMILES in place.

In [6]:
smiles_before = df["canonical_smiles"].copy()

df = standardize_molecules(df)

changed = (smiles_before != df["canonical_smiles"]).sum()
print(f"Standardized {changed} out of {len(df)} SMILES.")

Standardizing: 100%|██████████| 16864/16864 [01:54<00:00, 147.72it/s]

Standardized 2509 out of 16864 SMILES.


### **5. Deduplication**

After standardization, some molecules that had different SMILES representations now match. We group by `(target, SMILES)` and keep only the row with the lowest (best) IC50 value.

In [7]:
rows_before = len(df)
df = deduplicate(df)
print(f"Dropped {rows_before - len(df)} duplicate rows.")

Dropped 5848 duplicate rows.


### **6. Compute pIC50**

Fill in missing `pchembl_value` entries by computing pIC50 = −log₁₀(IC50 × 10⁻⁹). Rows where the calculation is impossible (IC50 ≤ 0) are removed.

In [8]:
missing_before = df["pchembl_value"].isna().sum()

df = compute_pic50(df)

missing_after = df["pchembl_value"].isna().sum()
print(f"Filled {missing_before - missing_after} missing pchembl values.")

rows_before = len(df)
df = df.dropna(subset=["pchembl_value"])
print(f"Dropped {rows_before - len(df)} rows with invalid IC50 (≤ 0).")

Computing pIC50: 100%|██████████| 11016/11016 [00:00<00:00, 34167.11it/s]

Filled 2093 missing pchembl values.
Dropped 4 rows with invalid IC50 (≤ 0).


### **7. Save Processed Data**

Export the cleaned dataset to CSV with a per-target row count summary.

In [9]:
save_cleaned_data(df, OUTPUT_PATH, targets=TARGETS)

Saved 11012 rows to /home/stanuch/Dev/TaniMol/data/processed/cleaned_activities.csv

Rows per target:
- PARP1 (CHEMBL3105): 4828
- PARP2 (CHEMBL5366): 879
- ATR (CHEMBL5024): 2511
- ATM (CHEMBL3797): 966
- DNA-PKcs (CHEMBL3142): 1828
